In [1]:
# Importy
import pandas as pd
import json
import os
from sentence_transformers import SentenceTransformer, util
from tqdm import tqdm

C:\Users\chlip\anaconda3\envs\bim-nlp\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Dane IFC
ifc_path = "../data/ifc_objects.csv"
ifc_df = pd.read_csv(ifc_path)
print(f"Wczytano {len(ifc_df)} obiektów IFC")

Wczytano 7 obiektów IFC


In [9]:
# Dane z bSDD
with open("../data/bsdd_cleared.json", encoding="utf-8") as f:
    bsdd_raw = json.load(f)
    
    bsdd_rows = []
    for group in bsdd_raw:
        for cls in group["classes"]:
            bsdd_rows.append({
            "dictionary_name": group["dictionary_name"],
            "class_code": cls.get("class_code", ""),
            "class_name": cls.get("class_name", ""),
            "class_description": cls.get("class_description", "")
            })

bsdd_df = pd.DataFrame(bsdd_rows)
print(f"Wczytano {len(bsdd_df)} klas bSDD")

Wczytano 3031 klas bSDD


In [10]:
# Mapowanie IFC
ifc_to_keywords = {
"IFCWALL": ["wall"],
"IFCDOOR": ["door"],
"IFCWINDOW": ["window"],
"IFCSLAB": ["slab", "floor", "roof"],
"IFCSTAIR": ["stair", "step"],
"IFCCOLUMN": ["column", "pillar"],
"IFCBEAM": ["beam"],
"IFCFURNISHINGELEMENT": ["furniture", "chair", "table", "shelf"],
"IFCRAILING": ["railing", "barrier"],
}

In [11]:
# Model
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

In [12]:
# Przygotowanie danych
bsdd_df["full_text"] = (
bsdd_df["class_code"].astype(str)
+ " — "
+ bsdd_df["class_name"].fillna("")
+ " — "
+ bsdd_df["class_description"].fillna("")
)

In [21]:
# Klasyfikacja
from sentence_transformers import util
from tqdm import tqdm

results = []

for i, row in tqdm(ifc_df.iterrows(), total=len(ifc_df)):
    ifc_type = row["IfcType"]
    ifc_text = str(row["Text"])
    ifc_name = str(row.get("Name", ""))

    # Słowa kluczowe dla danego typu IFC
    keywords = ifc_to_keywords.get(ifc_type.upper(), [])
    
    # Filtrowanie klas bSDD na podstawie słów kluczowych
    filtered_bsdd = bsdd_df[
        bsdd_df["class_name"].str.lower().str.contains("|".join(keywords), na=False)
        | bsdd_df["class_description"].str.lower().str.contains("|".join(keywords), na=False)
    ]
    
    if filtered_bsdd.empty: 
        continue
    
    # Embeddingi i podobieństwa
    ifc_embedding = model.encode([ifc_name + " " + ifc_text], convert_to_tensor=True)
    bsdd_embeddings = model.encode(filtered_bsdd["full_text"].tolist(), convert_to_tensor=True)
    
    similarities = util.cos_sim(ifc_embedding, bsdd_embeddings)[0]
    best_idx = similarities.argmax().item()
    best_score = similarities[best_idx].item()
    
    best_class = filtered_bsdd.iloc[best_idx]
    
    results.append({
        "GlobalId": row["GlobalId"],
        "IfcType": ifc_type,
        "Name": ifc_name,
        "Opis_IFC": ifc_text[:300],
        "Kod_bSDD": best_class["class_code"],
        "Nazwa_klasy_bSDD": best_class["class_name"],
        "Słownik": best_class["dictionary_name"],
        "Podobieństwo": round(best_score, 4)
    })

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:02<00:00,  2.54it/s]


In [22]:
# Wyniki
results_df = pd.DataFrame(results)
results_df.to_csv("../results/mapped_ifc_to_bsdd_filtered.csv", index=False)
print("Zapisano wynik do mapped_ifc_to_bsdd_filtered.csv")

Zapisano wynik do mapped_ifc_to_bsdd_filtered.csv


In [23]:
# Wyświetlanie bezpośrednio pod komórką
print("Zapisano wynik do mapped_ifc_to_bsdd_filtered.csv")
display(results_df)

Zapisano wynik do mapped_ifc_to_bsdd_filtered.csv


,GlobalId,IfcType,Name,Opis_IFC,Kod_bSDD,Nazwa_klasy_bSDD,Słownik,Podobieństwo
0,3zR0BOEcLADRKln4HYporH,IFCSLAB,floor,"A solid, site-cast concrete floor, providing a...",L-NCC,Flooring,CCI,0.5247
1,1AQAupaRP1txwK1AGiN61V,IFCWALL,house - outer wall - house right front,"A solid outer wall, forming the right front si...",EC000073,Junction box for wall duct,ETIM,0.5455
2,3wdauVJT5Fx9drrREiDqA$,IFCWALL,house - outer wall - house right back,"A solid outer wall, forming the right back sid...",EC000073,Junction box for wall duct,ETIM,0.5408
3,0OfZwWc8j9QP5uX8xPTxDH,IFCWALL,house - outer wall - house left,"A solid outer wall, forming the left side of t...",EC000073,Junction box for wall duct,ETIM,0.5251
4,1uS5vfZPn9R8PlAaVd73on,IFCWALL,plumbing wall,A wall designed to house and protect plumbing ...,EC000073,Junction box for wall duct,ETIM,0.6043
5,0ZTBBPo6f6bxqV2K7Oelrq,IFCSLAB,house - roof - slab left,A roof slab that's got it all covered. IsExter...,Ac_15_50_73,Roof surveying,Uniclass,0.6572
6,12UVOn4wvAJPMUExKdZLb8,IFCSLAB,house - roof - slab right,A roof slab that's got it all covered. IsExter...,Ac_15_50_73,Roof surveying,Uniclass,0.6685
